In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras


In [ ]:
mnist=tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test)=mnist.load_data()

In [ ]:
#normalizes color values to 0-1
x_train=tf.keras.utils.normalize(x_train, axis=1)
x_test=tf.keras.utils.normalize(x_test, axis=1)

In [ ]:
x_train=x_train.reshape(-1,28*28)   #flatten images
x_test=x_test.reshape(-1,28*28)

model = keras.Sequential([
    keras.Input(shape=(28*28,)), # Feature layer
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(x_train, y_train,epochs=3)

model.save('handwritten.keras')




In [ ]:
loss, accuracy = model.evaluate(x_test,y_test)
print(loss)
print(accuracy)

In [ ]:
from google.colab import files
import zipfile
import io
import shutil
import tempfile
from pathlib import Path

uploaded = files.upload()

extract_dir = tempfile.mkdtemp(prefix="mnist_upload_")

for filename, payload in uploaded.items():
    print(f"Uploaded file: {filename}")
    name = str(filename).lower()
    if name.endswith(".zip"):
        archive = io.BytesIO(payload) if isinstance(payload, (bytes, bytearray)) else filename
        with zipfile.ZipFile(archive, "r") as zip_ref:
            zip_ref.extractall(extract_dir)
    else:
        target = Path(extract_dir) / Path(str(filename)).name
        if isinstance(payload, (bytes, bytearray)):
            target.write_bytes(payload)
        else:
            shutil.copy(str(filename), target)

print(f"Contents extracted to temp dir: {extract_dir}")

In [ ]:
from pathlib import Path

image_paths = [
    p for p in Path(extract_dir).rglob("*")
    if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
]
print(f"Found {len(image_paths)} image files")
for path in image_paths[:20]:
    print(path.name)

In [ ]:
import shutil

for img_path in image_paths:
    try:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)  # Read as grayscale
        if img is None:
            continue

        img = cv2.resize(img, (28, 28))

        # invert colors
        img = cv2.bitwise_not(img)

        # Normalize and flatten the image
        img = img.astype("float32") / 255.0
        img = img.flatten()
        img = img.reshape(1, 784)

        prediction = model.predict(img)
        print(f"This digit is probably a {np.argmax(prediction)}")

        plt.imshow(img.reshape(28, 28), cmap=plt.cm.binary)
        plt.title(f"Predicted: {np.argmax(prediction)}")
        plt.show()
    except Exception as e:
        print(f"Error processing file {img_path}: {e}")

# Cleanup temp extraction directory so image trees are not persisted.
shutil.rmtree(extract_dir, ignore_errors=True)